In [21]:
pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [22]:
# Apple Inc. → tecnología, Microsoft Corporation → tecnología, Exxon Mobil Corporation → energía, SPDR Gold Shares → oro
#SPDR S&P 500 ETF Trust → mercado
import yfinance as yf 
import numpy as np
import pandas as pd
tickers = ["AAPL", "MSFT", "XOM", "GLD", "SPY"]

In [23]:
data = yf.download(tickers, start="2020-01-01", end="2026-01-01")
print(data.columns) # lo ideal es revisar primero el nombre de las columnas.

[*********************100%***********************]  5 of 5 completed

MultiIndex([( 'Close', 'AAPL'),
            ( 'Close',  'GLD'),
            ( 'Close', 'MSFT'),
            ( 'Close',  'SPY'),
            ( 'Close',  'XOM'),
            (  'High', 'AAPL'),
            (  'High',  'GLD'),
            (  'High', 'MSFT'),
            (  'High',  'SPY'),
            (  'High',  'XOM'),
            (   'Low', 'AAPL'),
            (   'Low',  'GLD'),
            (   'Low', 'MSFT'),
            (   'Low',  'SPY'),
            (   'Low',  'XOM'),
            (  'Open', 'AAPL'),
            (  'Open',  'GLD'),
            (  'Open', 'MSFT'),
            (  'Open',  'SPY'),
            (  'Open',  'XOM'),
            ('Volume', 'AAPL'),
            ('Volume',  'GLD'),
            ('Volume', 'MSFT'),
            ('Volume',  'SPY'),
            ('Volume',  'XOM')],
           names=['Price', 'Ticker'])


In [24]:
precios = data["Close"]
precios

Ticker,AAPL,GLD,MSFT,SPY,XOM
Date,,,,,
2020-01-02,72.400528,143.949997,152.158401,296.888245,53.306408
2020-01-03,71.696632,145.860001,150.263733,294.640076,52.877857
2020-01-06,72.267952,147.389999,150.652130,295.764130,53.283859
2020-01-07,71.928062,147.970001,149.278519,294.932495,52.847775
2020-01-08,73.085098,146.860001,151.656326,296.504333,52.050808
...,...,...,...,...,...
2025-12-24,273.554016,411.929993,486.908630,688.499695,118.430618
2025-12-26,273.144409,416.739990,486.599365,688.429871,118.321342
2025-12-29,273.504089,398.600006,485.990753,685.976562,119.731941


In [25]:
# Cuenta cuántos valores faltantes hay por cada activo y solo aplicados al precio que es lo que se necesita
print(precios.isnull().sum())

# indica el porcentaje de datos faltantes
print(precios.isnull().mean() * 100)

Ticker
AAPL    0
GLD     0
MSFT    0
SPY     0
XOM     0
dtype: int64
Ticker
AAPL    0.0
GLD     0.0
MSFT    0.0
SPY     0.0
XOM     0.0
dtype: float64


In [26]:
# Los precios solos no sirven de nada, ahora se van a transformar en rendimientos y se calculará el rendimiento porcentual diario
rendimientos = precios.pct_change()

In [27]:
rendimientos # en  este caso es completamente normal que el primer dato tenga NaN, ya que no hay día anterior.

Ticker,AAPL,GLD,MSFT,SPY,XOM
Date,,,,,
2020-01-02,NaN,NaN,NaN,NaN,NaN
2020-01-03,-0.009722,0.013269,-0.012452,-0.007572,-0.008039
2020-01-06,0.007969,0.010490,0.002585,0.003815,0.007678
2020-01-07,-0.004703,0.003935,-0.009118,-0.002812,-0.008184
2020-01-08,0.016086,-0.007502,0.015929,0.005329,-0.015080
...,...,...,...,...,...
2025-12-24,0.005324,-0.004134,0.002403,0.003518,-0.001675
2025-12-26,-0.001497,0.011677,-0.000635,-0.000101,-0.000923
2025-12-29,0.001317,-0.043528,-0.001251,-0.003564,0.011922


In [28]:
# para eliminar campos NaN
rendimientos = rendimientos.dropna()

In [29]:
rendimientos.head()

Ticker,AAPL,GLD,MSFT,SPY,XOM
Date,,,,,
2020-01-03,-0.009722,0.013269,-0.012452,-0.007572,-0.008039
2020-01-06,0.007969,0.010490,0.002585,0.003815,0.007678
2020-01-07,-0.004703,0.003935,-0.009118,-0.002812,-0.008184
2020-01-08,0.016086,-0.007502,0.015929,0.005329,-0.015080
2020-01-09,0.021241,-0.005652,0.012493,0.006781,0.007656


In [30]:
rendimientos.shape # shape indica las dimensiones

(1507, 5)

In [31]:
rendimientos.mean() # rendimiento promedio diario (esperado) y es el retorno aritmético real

Ticker
AAPL    0.001078
GLD     0.000725
MSFT    0.000939
SPY     0.000636
XOM     0.000750
dtype: float64

In [32]:
rendimientos.std() # mide la volatilidad de las acciones. 

Ticker
AAPL    0.020038
GLD     0.010295
MSFT    0.018613
SPY     0.013070
XOM     0.020667
dtype: float64

In [33]:
rendimientos.corr()

Ticker,AAPL,GLD,MSFT,SPY,XOM
Ticker,,,,,
AAPL,1.000000,0.103276,0.709151,0.783912,0.291231
GLD,0.103276,1.000000,0.092170,0.131937,0.082316
MSFT,0.709151,0.092170,1.000000,0.803279,0.236028
SPY,0.783912,0.131937,0.803279,1.000000,0.505069
XOM,0.291231,0.082316,0.236028,0.505069,1.000000


In [34]:
pesos = np.array([0.2, 0.2, 0.2, 0.2, 0.2]) # aquí se define cuanto dinero se pone a cada acción
portafolio = rendimientos.values @ pesos

np.std(portafolio) # volatilidad del portafolio

np.float64(0.012040061736528234)

In [35]:
np.mean(portafolio)

np.float64(0.0008255178248308487)

In [36]:
# El portafolio diversificado logra reducir la volatilidad total en comparación
# con los activos individuales, evidenciando los beneficios de combinar activos
# con diferentes niveles de correlación.
#
# La inclusión de activos como GLD (baja correlación) y XOM (diversificación sectorial)
# permite mitigar el riesgo sin eliminar completamente el potencial de retorno.
#
# Aunque el portafolio no maximiza el rendimiento, sí mejora el equilibrio
# riesgo-retorno, lo que representa un enfoque más robusto desde la perspectiva
# de gestión de inversiones.